# KG1 V1244 CoT-safe — TREINO REAL (notebook LIMPO)

**Rode na ordem: [1] deps → [2] token → [3] treino.**

⚠️ **MANTENHA O COLAB VIVO as ~6.6h:** não feche a aba, não deixe o PC dormir.
(Sessão ociosa = Colab desconecta = treino morre.)

- mb=2 · 160 steps · warmstart 086 · checkpoints sobem no HF (40/80/120 + final)
- Logs ao vivo: https://huggingface.co/datasets/felipesp1983/kg1-live-logs/tree/main/colab


In [ ]:
import os, subprocess, sys, glob
print('[1/5] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'], check=True)
os.chdir('/content/kg1')
print('[2/5] deps base', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','bitsandbytes','safetensors','huggingface_hub','hf_xet','einops','ninja'], check=False)
print('[3/5] pin torch 2.10 cu126 (~2-3min; p/ casar o wheel do mamba)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.10.0','torchvision==0.25.0','torchaudio==2.10.0','--index-url','https://download.pytorch.org/whl/cu126'], check=False)  # trio casado (senao torchvision::nms quebra)
import torch
assert torch.cuda.is_available() and torch.version.cuda, f'torch SEM CUDA apos pin (v={torch.__version__}). Reinicie e rode de novo.'
py=f"cp{sys.version_info.major}{sys.version_info.minor}"
tmm='.'.join(torch.__version__.split('+')[0].split('.')[:2])
cu='cu'+((torch.version.cuda or '12').split('.')[0])
abi='TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
print(f'[env] {py} torch{tmm} {cu} cxx11abi{abi} cuda_ok={torch.cuda.is_available()}', flush=True)
print('[4/5] mamba-ssm 2.3.1 (WHEEL PRONTO ~10s; fallback source)', flush=True)
url=f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{tmm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
print('  tentando wheel:', url, flush=True)
if subprocess.run([sys.executable,'-m','pip','install','--no-deps',url]).returncode!=0:
    print('  >>> wheel nao casou -> compilando do source (~25min)', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'], check=False)
print('[5/5] causal-conv1d 1.6.1 (tenta WHEEL cacheado no HF ~10s; senao compila ~5min)', flush=True)
wtag=f"{py}_torch{tmm}_cu{(torch.version.cuda or 'na').replace('.','')}"
causal_ok=False
try:
    from google.colab import userdata
    for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
        try:
            tv=userdata.get(k)
            if tv: os.environ.setdefault('HF_TOKEN',tv); break
        except Exception: pass
    from huggingface_hub import snapshot_download
    d=snapshot_download('felipesp1983/kg1-wheels', repo_type='dataset', allow_patterns=f'{wtag}/causal*', token=os.environ.get('HF_TOKEN'))
    cw=glob.glob(f'{d}/{wtag}/causal*.whl')
    if cw and subprocess.run([sys.executable,'-m','pip','install','--no-deps']+cw).returncode==0:
        print('  causal via WHEEL cacheado (HF) ~10s', flush=True); causal_ok=True
except Exception as e:
    print('  sem cache causal (', str(e)[:70], ')', flush=True)
if not causal_ok:
    print('  compilando causal do source (~5min)', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'], check=False)
try:
    import mamba_ssm, causal_conv1d
    print('IMPORT OK: mamba_ssm + causal_conv1d', flush=True)
except Exception as e:
    raise RuntimeError('FALHA import mamba_ssm/causal_conv1d apos install: '+str(e)[:200]+' -> reinicie o runtime e rode de novo.')
print('DEPS OK', flush=True)

In [ ]:
from google.colab import userdata
import os
for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
    try:
        v=userdata.get(k)
        if v: os.environ['HF_TOKEN']=v; os.environ['HF_KEY']=v; break
    except Exception: pass
assert os.environ.get('HF_TOKEN'), 'Defina HF_KEY no Colab Secrets (com escrita)'
print('HF token OK', flush=True)

In [ ]:
# ============================================================================
# [3] TREINO REAL  (mb=2, 160 steps)  — rode DEPOIS de [1] deps e [2] token
# ============================================================================
import os, sys, subprocess, time
os.chdir('/content/kg1')
subprocess.run(['git','fetch','-q','origin','claude/v1244-cot-safe'])
subprocess.run(['git','reset','--hard','origin/claude/v1244-cot-safe'])
print('HEAD:', subprocess.run(['git','rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

# fixes + observabilidade
os.environ['MODEL_DEVICE_MAP'] = 'cpu_then_cuda'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KG1_DEBUG_MICRO'] = '1'
os.environ['KG1_TRACE_LAYER_TYPES'] = '1'
os.environ['KG1_SCORE_LIVE'] = '0'                       # geracao HF inviavel no Mamba (juiz = Notebook B)
os.environ['SCORE_PROXY_EVAL_MAX_EXAMPLES'] = '48'
os.environ['UPLOAD_CHECKPOINTS_DURING_TRAINING'] = '1'   # sobe checkpoint-40/80/120 (Notebook B avalia + sobrevive a crash)

# modelo + dados
os.environ['MODEL_NAME'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
os.environ['MODEL_REVISION'] = 'cbd3fa9f933d55ef16a84236559f4ee2a0526848'
os.environ['ATTN_IMPLEMENTATION'] = 'eager'; os.environ['GRADIENT_CHECKPOINTING'] = '1'
DATA = 'artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'
os.environ['DATA_FILE'] = DATA; os.environ['VAL_FILE'] = DATA

# warmstart 086 + LoRA + receita
os.environ['INIT_ADAPTER_REPO'] = 'felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086'
os.environ['INIT_ADAPTER_REVISION'] = 'f4134a6d223249d27be2f1c5d94ed59d118d1ce5'
os.environ['REQUIRE_INIT_ADAPTER'] = '1'
os.environ['LORA_R'] = '32'; os.environ['LORA_ALPHA'] = '32'
os.environ['BATCH_SIZE'] = '32'; os.environ['MICRO_BATCH_SIZE'] = '2'   # mb=2 (validado 76.1GiB)
os.environ['LEARNING_RATE'] = '5e-6'; os.environ['FINAL_LEARNING_RATE'] = '1e-6'
os.environ['BOXED_PAYLOAD_LOSS_WEIGHT'] = '1.0'; os.environ['REQUIRE_OFFSET_MASK'] = '1'
os.environ['OUTPUT_REPO'] = 'felipesp1983/kg1-v1244-cot-candidate'; os.environ['UPLOAD_TO_HF'] = '1'

# TREINO REAL: 160 steps, 6 epocas, eval/save @40
os.environ['MAX_STEPS'] = '160'; os.environ['NUM_EPOCHS'] = '6'
os.environ['EVAL_EVERY_STEPS'] = '40'; os.environ['SAVE_EVERY_STEPS'] = '40'

# gates de integridade (sha LF do Colab)
SHA = '8d90bb2de38ab13a438092036b039469c3b5f73ac0f6fdf1f7e27a0b5bde8c2c'
os.environ['EXPECTED_TRAIN_SHA256'] = SHA; os.environ['EXPECTED_VAL_SHA256'] = SHA
os.environ['MIN_TRAIN_EXAMPLES'] = '500'; os.environ['MIN_VAL_EXAMPLES'] = '100'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES'] = '500'; os.environ['MIN_TOKENIZED_VAL_EXAMPLES'] = '100'
os.environ['SCORE_TRAJECTORY_CHECK'] = '0'

# live-log + watchdog
os.environ['KG1_LIVE_LOG_HF_REPO'] = 'felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE'] = 'dataset'
os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD'] = '1'
os.environ['KG1_WATCHDOG_STALE_SECONDS'] = '2700'
os.environ['KG1_LIVE_LOG_UPLOAD_EVERY'] = '20'

# RUN
os.environ['RUN_ID'] = 'v1244_train_' + time.strftime('%Y%m%d_%H%M%S')
print('TREINO REAL mb=2 160 steps | RUN_ID=', os.environ['RUN_ID'], flush=True)
r = subprocess.run([sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/hf_job_train_v90.py'])
print('RETURN_CODE=', r.returncode, flush=True)
